# Databricks RAG -- Mosaic AI Production Layer (V2)

**Author:** Ravi Amaraweera -- Senior Data Architect / Analytics Engineer  
**Stack:** Databricks Vector Search | Mosaic AI Agent Framework | MLflow | LangChain  
**Builds on:** `databricks_rag_vector_search.ipynb` (V1 -- LangChain prototype)  
**Data:** Apache Airflow open-source docs (Apache 2.0 licence)  
**Estimated cost:** $5-$15 within Databricks $400 free trial credits

---

## What this notebook adds over V1

V1 built the RAG chain inside a notebook -- useful for exploration but not consumable
outside Databricks. V2 adds the **Mosaic AI production layer**:

| Capability | Tool | What it gives you |
|---|---|---|
| Model versioning | MLflow + Unity Catalog | Every chain iteration tracked and reproducible |
| Live REST API | agents.deploy | Any app can call the RAG agent over HTTP |
| Scale-to-zero serving | Serverless Model Serving | Endpoint costs nothing when idle |
| Shareable Review App | Mosaic AI Review App | Colleagues test the agent without Databricks access |
| Observability | MLflow Tracing | Every retrieval and LLM call recorded for debugging |
| Quality scoring | Agent Evaluation | LLM judges score groundedness and relevance |

## Prerequisites

Run `databricks_rag_vector_search.ipynb` first so that:
- Vector Search endpoint `rag-portfolio-endpoint` is ONLINE
- Vector Search index `rag_portfolio.doc_search.airflow_docs_index` is ONLINE

## Execution order

Run cells top-to-bottom. Cell 18 (Cleanup) is commented out by design.

---
## Cell 1 -- Install Mosaic AI Packages

Installs the two extra packages V1 did not need:
- `databricks-agents`: provides `agents.deploy()` for Model Serving deployment
- `mlflow[databricks]`: adds Databricks-specific MLflow tracing and agent evaluation

> The kernel restarts automatically -- start from Cell 2.

In [0]:
%pip install databricks-agents mlflow[databricks] databricks-langchain langchain langchain-text-splitters
# Kernel restarts automatically so newly installed packages are importable.
dbutils.library.restartPython()

---
## Cell 2 -- Set Shared Variables

All configuration constants in one place. Customise these if your catalog, schema, or
endpoint names differ from V1.

- `MODEL_NAME`: fully-qualified Unity Catalog path for model registration
- `VS_ENDPOINT` / `VS_INDEX`: must match what was created in V1
- `LLM_ENDPOINT`: Foundation Model API endpoint used for generation
- `EMBED_MODEL`: embedding model -- must match the model used when building the V1 index

In [0]:
CATALOG      = "rag_portfolio"
SCHEMA       = "doc_search"
MODEL_NAME   = f"{CATALOG}.{SCHEMA}.airflow_rag_agent"  # Unity Catalog model registry path
VS_ENDPOINT  = "rag-portfolio-endpoint"                   # Vector Search endpoint (from V1)
VS_INDEX     = f"{CATALOG}.{SCHEMA}.airflow_docs_index"  # Vector Search index (from V1)
SERVING_EP   = "airflow-rag-serving-endpoint"             # new Model Serving endpoint name
LLM_ENDPOINT = "databricks-gpt-oss-20b"                  # cheapest Foundation Model API option
EMBED_MODEL  = "databricks-bge-large-en"                  # must match V1 index embedding model

import mlflow

# Tell MLflow to use Unity Catalog as the model registry.
# Required for agents.deploy() in Cell 9.
mlflow.set_registry_uri("databricks-uc")

print("Variables set. MLflow registry pointed to Unity Catalog.")
print(f"Model will be registered as: {MODEL_NAME}")

---
## Cell 3 -- Create an MLflow Experiment

An MLflow Experiment groups all model runs (iterations) for this project.
Each time you log the model in Cell 7 a new Run is created inside this experiment.

Think of it as version history for your chain:
- Run 1: initial chain, average score 3.2/5
- Run 2: improved prompt, average score 4.1/5

View all runs: left sidebar -> Experiments -> airflow_rag_v2

In [0]:
# Set the active MLflow experiment.
# If it does not exist yet, MLflow creates it automatically.
experiment_name = "/Shared/airflow_rag_v2"
mlflow.set_experiment(experiment_name)

print(f"MLflow experiment set: {experiment_name}")
print("View it: left sidebar -> Experiments -> airflow_rag_v2")

---
## Cell 4 -- Reconnect to Vector Search and Rebuild the Chain

Re-establishes the LangChain retriever and QA chain after the kernel restart.
All in-memory Python objects from V1 were lost when the kernel restarted in Cell 1.
This cell reconstructs the chain so it can be wrapped by MLflow in Cell 5.

`SimpleRetrievalQA` is a lightweight replacement for the removed
`langchain.chains.RetrievalQA` -- it performs the same retrieve-prompt-LLM pipeline.

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain import DatabricksVectorSearch, ChatDatabricks
from langchain_core.prompts import PromptTemplate


class SimpleRetrievalQA:
    # Minimal drop-in for the deprecated langchain RetrievalQA.
    # Pipeline: retrieve top-k chunks -> format context -> call LLM -> return answer.
    def __init__(self, llm, retriever, prompt):
        self.llm = llm
        self.retriever = retriever
        self.prompt = prompt

    def invoke(self, inputs):
        query = inputs["query"]
        # Retrieve the most semantically similar document chunks for this query.
        docs = self.retriever.invoke(query)
        # Join chunk text into a single context block for the prompt.
        context = "\n\n".join(doc.page_content for doc in docs)
        # Format the full prompt and send it to the LLM.
        formatted = self.prompt.format(context=context, question=query)
        response = self.llm.invoke(formatted)
        return {"result": response.content, "source_documents": docs}


# temperature=0: deterministic responses -- best for factual Q&A.
# Foundation Model API authenticates via the cluster token -- no extra API key needed.
llm = ChatDatabricks(
    endpoint=LLM_ENDPOINT,
    temperature=0,
    max_tokens=512
)

# Reconnect to the existing Vector Search index.
vs_client = VectorSearchClient()
vectorstore = DatabricksVectorSearch(
    endpoint=VS_ENDPOINT,
    index_name=VS_INDEX,
    columns=["text", "source", "doc_type"]  # metadata columns returned per chunk
)
# k=4: return the top-4 most semantically similar chunks per query.
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Constraining the LLM to use only the provided context is the primary
# hallucination-mitigation technique in a RAG system.
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are a technical assistant for Apache Airflow.\n"
        "Answer the question using ONLY the provided context.\n"
        "If the context is insufficient, say: I do not have enough context.\n"
        "\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    )
)

# Assemble the QA chain from the three components above.
qa_chain = SimpleRetrievalQA(llm=llm, retriever=retriever, prompt=prompt_template)

print("Chain rebuilt successfully.")
print(f"  LLM endpoint  : {LLM_ENDPOINT}")
print(f"  Vector Search : {VS_INDEX}")
print(f"  Retriever k   : 4")

---
## Cell 5 -- Wrap the Chain in an MLflow PyFunc Model Class

MLflow needs the chain packaged as a **Python model file** (not an in-memory object)
so it can be serialised, versioned, and deployed to Model Serving.

`RAGChainModel` extends `mlflow.pyfunc.ChatModel`, which means:
- MLflow knows it accepts Chat API-format requests
- It produces OpenAI-compatible Chat API-format responses
- The serving endpoint automatically exposes a `/invocations` REST endpoint

The file is written to `/tmp/rag_chain_model.py` and bundled with the model artefact.

In [0]:
# Write the RAGChainModel class to a standalone Python file.
# MLflow requires a file path (not a class instance) for code-based model logging.
# This file is packaged with the model artefact and executed on the serving endpoint.

model_code = '''
import mlflow
from mlflow.pyfunc import ChatModel
from mlflow.types.llm import ChatCompletionResponse, ChatMessage, ChatChoice

# Configuration -- must match Cell 2 values.
CATALOG      = "rag_portfolio"
SCHEMA       = "doc_search"
VS_ENDPOINT  = "rag-portfolio-endpoint"
VS_INDEX     = f"{CATALOG}.{SCHEMA}.airflow_docs_index"
LLM_ENDPOINT = "databricks-gpt-oss-20b"
EMBED_MODEL  = "databricks-bge-large-en"


class RAGChainModel(ChatModel):
    """
    Production RAG chain wrapped as an MLflow ChatModel.
    Inheriting from ChatModel makes this class compatible with:
      - Unity Catalog model registry
      - Mosaic AI Model Serving (agents.deploy)
      - Mosaic AI Agent Evaluation
      - The Review App shareable chat UI
    """

    def load_context(self, context):
        # Called once at model load time on the serving endpoint.
        # Sets up the retriever and LLM -- avoids reinitialising on every request.
        from databricks_langchain import DatabricksVectorSearch, ChatDatabricks
        from langchain_core.prompts import PromptTemplate

        class SimpleRetrievalQA:
            def __init__(self, llm, retriever, prompt):
                self.llm = llm
                self.retriever = retriever
                self.prompt = prompt
            def invoke(self, inputs):
                docs = self.retriever.invoke(inputs["query"])
                ctx  = "\n\n".join(d.page_content for d in docs)
                fmt  = self.prompt.format(context=ctx, question=inputs["query"])
                resp = self.llm.invoke(fmt)
                return {"result": resp.content, "source_documents": docs}

        llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0, max_tokens=512)
        vectorstore = DatabricksVectorSearch(
            endpoint=VS_ENDPOINT,
            index_name=VS_INDEX,
            columns=["text", "source", "doc_type"]
        )
        retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
        prompt = PromptTemplate(
            input_variables=["context", "question"],
            template=(
                "You are an Airflow technical assistant.\n"
                "Answer using ONLY the provided context.\n"
                "If insufficient, say: I do not have enough context.\n"
                "\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
            )
        )
        # Store chain as instance attribute -- accessible by predict() on every request.
        self.qa_chain = SimpleRetrievalQA(llm=llm, retriever=retriever, prompt=prompt)

    def predict(self, context, messages, params=None):
        # Called on every inference request.
        # Extracts the user question from the last message and runs the RAG chain.
        user_question = messages[-1].content
        result = self.qa_chain.invoke({"query": user_question})
        answer = result["result"]
        # Return OpenAI-compatible ChatCompletionResponse.
        return ChatCompletionResponse(
            choices=[ChatChoice(message=ChatMessage(role="assistant", content=answer))]
        )
'''

model_path = "/tmp/rag_chain_model.py"
with open(model_path, "w") as f:
    f.write(model_code.lstrip())

print(f"RAGChainModel written to: {model_path}")

---
## Cell 6 -- Test the Wrapper Locally Before Logging

Smoke-tests `RAGChainModel.predict()` in the notebook to catch import errors or
logic issues before committing to a full MLflow run.

Best practice: always run a local test before logging to MLflow.

In [0]:
from mlflow.types.llm import ChatMessage
from rag_chain_model import RAGChainModel  # imported from the file written in Cell 5

# Instantiate and attach the qa_chain built in Cell 4.
# On a real serving endpoint, load_context() handles this automatically.
test_model = RAGChainModel()
test_model.qa_chain = qa_chain  # reuse the already-built chain

result = test_model.predict(
    context=None,
    messages=[ChatMessage(role="user", content="What are best practices for Airflow DAG design?")]
)

print("Local test passed. Answer:")
print(result.choices[0].message.content)

---
## Cell 7 -- Log the Model to MLflow

Logs the RAG chain as a versioned MLflow run inside the experiment from Cell 3.

Key concepts:
- `mlflow.start_run()`: opens a new experiment run with a unique run_id
- `mlflow.pyfunc.log_model()`: packages the model file and dependencies
- `resources`: declares Databricks service dependencies for auto-IAM configuration
- `input_example`: used to generate the REST API schema for the serving endpoint

In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint
import mlflow

# Sample input in Chat API format.
# MLflow uses this to infer the serving endpoint input/output schema automatically.
input_example = {
    "messages": [{"role": "user", "content": "How do I create an Airflow DAG?"}]
}

with mlflow.start_run(run_name="airflow_rag_codebased") as run:
    model_info = mlflow.pyfunc.log_model(
        artifact_path="rag_chain",
        python_model="/tmp/rag_chain_model.py",   # file path (not a class instance)
        resources=[
            # Declare all Databricks services this model depends on.
            # MLflow uses this to auto-configure IAM permissions on the endpoint.
            DatabricksVectorSearchIndex(index_name=VS_INDEX),
            DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT),
            DatabricksServingEndpoint(endpoint_name=EMBED_MODEL),
        ],
        # Pin exact package versions for reproducibility on the serving endpoint.
        pip_requirements=[
            "databricks-vectorsearch",
            "langchain",
            "langchain-community",
            "databricks-langchain",
        ],
        input_example=input_example,
    )

RUN_ID = run.info.run_id
print(f"Model logged.")
print(f"  Run ID    : {RUN_ID}")
print(f"  Model URI : {model_info.model_uri}")

---
## Cell 8 -- Register the Model in Unity Catalog

Registers the logged MLflow artefact into the Unity Catalog model registry.
Each call creates a new version (Version 1, Version 2, ...).

Why register?
- Provides version history and rollback capability
- Models inherit Unity Catalog access controls
- `agents.deploy()` requires a registered model version number

View in: Catalog -> rag_portfolio -> doc_search -> airflow_rag_agent -> Versions

In [0]:
# Register the logged model from Cell 7 into Unity Catalog.
# Creates the model if it does not exist; adds a new version if it does.
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=MODEL_NAME
)

MODEL_VERSION = registered_model.version

print(f"Model registered in Unity Catalog")
print(f"  Full name : {MODEL_NAME}")
print(f"  Version   : {MODEL_VERSION}")
print(f"View in: Catalog -> rag_portfolio -> doc_search -> airflow_rag_agent")

---
## Cell 9 -- Deploy to a Model Serving Endpoint (scale-to-zero)

`agents.deploy()` is the Mosaic AI one-liner that provisions a full production deployment:
1. Creates a serverless Model Serving endpoint (CPU, auto-scales)
2. Enables scale-to-zero -- the endpoint costs nothing when idle
3. Creates a Review App -- shareable chat UI requiring no Databricks login
4. Configures authentication and access to declared resource dependencies

Cost note: unlike the Vector Search endpoint (~$0.28/hr always-on), the serving
endpoint has scale-to-zero so you only pay during active inference.

Wait time: 3-8 minutes. The Review App URL will not work until Cell 10 confirms READY.

In [0]:
from databricks import agents

print(f"Deploying model: {MODEL_NAME}  version: {MODEL_VERSION}")

# agents.deploy() provisions a serverless Model Serving endpoint with:
#   - scale_to_zero=True by default (no idle cost)
#   - A shareable Review App chat UI
#   - An /invocations REST endpoint compatible with the OpenAI Chat API
deployment = agents.deploy(
    input_schema='ChatCompletionRequest',
    model_name=MODEL_NAME,
    model_version=MODEL_VERSION,
)

print("Deployment started...")
print(f"  Serving endpoint : {deployment.endpoint_name}")
print(f"  Review App URL   : {deployment.review_app_url}")

---
## Cell 10 -- Wait for the Endpoint to Be Ready

Polls the Model Serving endpoint every 30 seconds until status is READY.
You can also watch progress in the Databricks UI: Serving tab -> endpoint name.

In [0]:
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
endpoint_name = deployment.endpoint_name

print(f"Waiting for endpoint '{endpoint_name}' to be ready...")

while True:
    state = w.serving_endpoints.get(name=endpoint_name).state
    ready = str(state.ready) if state else "UNKNOWN"
    print(f"  Status: {ready}")
    if "READY" in ready:
        print("Endpoint READY -- proceed to Cell 11")
        break
    time.sleep(30)

---
## Cell 11 -- Call the Live REST API

Queries the deployed endpoint using `mlflow.deployments.get_deploy_client("databricks")`
-- the same client any external application would use.

Portfolio takeaway: the RAG chain is now a live REST API. Any service that can send
an HTTP POST request can call it -- a web app, another Databricks job, or a data pipeline.

In [0]:
import mlflow.deployments

# The deployments client authenticates via the workspace token automatically.
deploy_client = mlflow.deployments.get_deploy_client("databricks")

test_questions = [
    "How do I set up task dependencies in an Airflow DAG?",
    "What are best practices for writing Airflow operators?",
]

for question in test_questions:
    response = deploy_client.predict(
        endpoint=endpoint_name,
        inputs={"messages": [{"role": "user", "content": question}]}
    )
    print(f"{'='*65}")
    print(f"Q: {question}")
    print(f"A: {response}")
    print()

---
## Cell 12 -- Add MLflow Tracing

MLflow Tracing records every step inside a RAG function call -- like a flight recorder
for your AI chain.

Why this matters: without tracing, when the chain gives a wrong answer you cannot tell
whether the retriever returned irrelevant chunks (retrieval problem) or the LLM misread
good chunks (generation problem).

With `@mlflow.trace`, each call creates a trace visible in:
Experiments -> airflow_rag_v2 -> Traces tab

Each trace shows:
- RETRIEVER span: which chunks were retrieved and their source files
- LLM span: the exact prompt sent and the response received
- Per-span latency (useful for SLA monitoring)

In [0]:
import mlflow

@mlflow.trace(name="airflow_rag_query")
def traced_rag_query(question: str) -> dict:
    # Wraps the RAG chain with MLflow tracing.
    # Every call creates a trace visible in the MLflow Traces tab.

    # RETRIEVER span: records which chunks were fetched and from which source files.
    with mlflow.start_span(name="retriever", span_type="RETRIEVER") as span:
        docs = retriever.invoke(question)
        span.set_inputs({"query": question})
        span.set_outputs({
            "num_docs_retrieved": len(docs),
            "sources": [d.metadata.get("source") for d in docs]
        })

    # LLM span: records the exact prompt and model response for debugging.
    with mlflow.start_span(name="llm_call", span_type="LLM") as span:
        result = qa_chain.invoke({"query": question})
        span.set_inputs({"query": question, "num_context_docs": len(docs)})
        span.set_outputs({"answer_length": len(result["result"])})

    return {
        "answer": result["result"],
        "sources": [d.metadata.get("source") for d in docs]
    }


# Run traced queries -- view each trace in the MLflow Traces tab.
traced_questions = [
    "How do I handle DAG failures in Airflow?",
    "What is the TaskFlow API?",
    "How do I use XComs between tasks?",
]

for q in traced_questions:
    output = traced_rag_query(q)
    print(f"Q: {q}")
    print(f"A: {output['answer'][:200]}...")
    print(f"   Sources: {output['sources']}")
    print()

print("Traces logged. View: Experiments -> airflow_rag_v2 -> Traces tab")

---
## Cell 13 -- Create the Evaluation Dataset

Agent Evaluation requires a golden dataset: questions paired with human-verified
expected answers (ground truth).

Each `expected_response` is what a domain expert says is the correct answer.
LLM judges compare the chain output against this ground truth.
Five pairs is sufficient to demonstrate the evaluation pattern.

In [0]:
import pandas as pd

# Golden evaluation dataset.
# request: the question sent to the RAG chain.
# expected_response: human-verified correct answer used by LLM judges.
eval_dataset = [
    {
        "request": "How do I set task dependencies in Airflow?",
        "expected_response": "Use the >> and << operators or set_downstream/set_upstream methods to define dependencies between tasks in an Airflow DAG."
    },
    {
        "request": "What is an Airflow Sensor?",
        "expected_response": "A Sensor is a special type of Operator that keeps running until a certain condition is met, such as a file appearing in S3."
    },
    {
        "request": "How do I retry failed Airflow tasks?",
        "expected_response": "Set retries and retry_delay parameters on the task, for example retries=3 and retry_delay=timedelta(minutes=5)."
    },
    {
        "request": "What is the Airflow XCom mechanism?",
        "expected_response": "XComs allow tasks to exchange small amounts of data using xcom_push to send and xcom_pull to receive values."
    },
    {
        "request": "How do I schedule an Airflow DAG to run daily?",
        "expected_response": "Set schedule_interval='@daily' or schedule_interval='0 0 * * *' in the DAG definition."
    },
]

eval_df = pd.DataFrame(eval_dataset)
print(f"Evaluation dataset created with {len(eval_df)} QA pairs")
display(eval_df)

In [0]:
# Pre-flight check: verify qa_chain is available before running evaluation.
# If this fails, re-run Cell 4 to rebuild the chain.
try:
    test = qa_chain.invoke({"query": "test"})
    print("qa_chain is alive -- ready to run evaluation")
except NameError:
    print("qa_chain is not defined -- re-run Cell 4 first")

---
## Cell 14 -- Run Agent Evaluation with LLM Judges

Runs the evaluation pipeline against every QA pair in the dataset.

Built-in Mosaic AI judges (using mlflow.evaluate with model_type='databricks-agent'):

| Judge | What it measures | Score |
|---|---|---|
| groundedness | Is the answer supported by context, not hallucinated? | 1-5 |
| relevance_to_query | Does the answer address the question? | 1-5 |
| retrieval_precision | Were the retrieved chunks relevant? | 1-5 |

The fallback scorer below uses word-overlap as a groundedness proxy.
Target: 4/5 or above for a strong portfolio demo. Takes 2-4 minutes.

In [0]:
import pandas as pd
import mlflow
import mlflow.genai

# Word-overlap groundedness scorer.
# Fallback when full Mosaic AI LLM judges are not available.
# Production: use mlflow.evaluate() with model_type='databricks-agent'.
def score_groundedness(inputs, outputs, expectations):
    response = str(outputs.get("response", "")).lower()
    expected = str(expectations.get("expected_response", "")).lower()
    expected_words = set(expected.split())
    response_words = set(response.split())
    overlap = len(expected_words & response_words)
    # Scale the overlap fraction to a 1-5 integer score.
    score = min(5, max(1, int((overlap / max(len(expected_words), 1)) * 5) + 1))
    return {"groundedness_score": score}


# Generate answers and score each question.
results = []
for row in eval_dataset:
    question = row["request"]
    output   = qa_chain.invoke({"query": question})
    answer   = output["result"]
    sources  = [d.metadata.get("source", "unknown") for d in output["source_documents"]]
    score_output = score_groundedness(
        inputs={"query": question},
        outputs={"response": answer},
        expectations={"expected_response": row["expected_response"]}
    )
    results.append({
        "question":           question,
        "answer":             answer,
        "groundedness_score": score_output["groundedness_score"],
        "sources":            sources
    })

results_df = pd.DataFrame(results)
print(f"Evaluation complete: {len(results_df)} questions scored.")
print(f"Average groundedness score: {results_df['groundedness_score'].mean():.1f}/5.0")

---
## Cell 15 -- View and Understand the Evaluation Results

Displays per-question scores and aggregate metrics.

Score interpretation:
- 5 -- Excellent: fully grounded, directly answers the question
- 4 -- Good: minor gaps but acceptable for production
- 3 -- Needs improvement: partially grounded
- 1-2 -- Poor: hallucinated or wrong chunk retrieved

Use these scores to identify which questions need prompt tuning or a higher k value.

In [0]:
print('=' * 55)
print('  RAG Evaluation -- Airflow Knowledge Base')
print('  Model: databricks_rag_mosaic_ai_v2')
print('=' * 55)

for _, row in results_df.iterrows():
    filled   = int(row['groundedness_score'])
    unfilled = 5 - filled
    bar = ('*' * filled) + ('-' * unfilled)
    print(f"\n  Q: {row['question'][:45]}...")
    print(f"     Score  : [{bar}] {filled}/5")
    print(f"     Source : {row['sources'][0].split('/')[-1] if row['sources'] else 'N/A'}")

print('\n' + '=' * 55)
print(f"  Average Score: {results_df['groundedness_score'].mean():.1f} / 5.0")
print('=' * 55)
print('Target: groundedness >= 4. Low scores = wrong chunks OR hallucinating LLM.')

---
## Cell 16 -- Explore the Review App

The Review App is a shareable chat UI created automatically alongside the serving endpoint.
No Databricks login is required -- suitable for sharing with stakeholders or clients.

Steps:
1. Copy the Review App URL from Cell 9 output
2. Open in any browser
3. Ask Airflow questions -- the UI shows the answer AND source chunks used
4. Screenshot the UI and add it to your GitHub README as social proof

In [0]:
print('=== Your Review App ===')
print(f'URL: {deployment.review_app_url}')
print()
print('Steps:')
print('  1. Open the URL in any browser (no Databricks login needed)')
print("  2. Ask: 'How do I set up task dependencies in an Airflow DAG?'")
print('  3. Screenshot the answer + source chunks panel')
print('  4. Add the screenshot to your GitHub README')
print()
print('The Review App is shareable -- send it to colleagues or clients as a live demo.')

---
## Cell 17 -- Version 2: Improve and Re-deploy

Demonstrates the MLOps iterative improvement cycle:

    Evaluate V1 -> identify weak questions -> improve prompt -> log V2 -> evaluate V2 -> compare

The improved prompt instructs the LLM to use expert Airflow terminology,
reference specific operators and parameters, and use bullet points for clarity.

After validating the improved answers, run Cells 5 -> 7 -> 8 again to register
Version 2 in Unity Catalog alongside Version 1.

In [0]:
# Improved prompt with more specific expert instructions.
# Changes:
#   - Requests specific Airflow operator names and parameters in answers
#   - Instructs the LLM to use bullet points for multi-step answers
improved_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are an expert Apache Airflow data engineering assistant.\n"
        "Answer the question using ONLY the provided context.\n"
        "If the context is insufficient, say: I do not have enough context.\n"
        "Mention specific Airflow operators, parameters, or code patterns when relevant.\n"
        "Use bullet points for multi-step answers.\n\n"
        "Context:\n{context}\n\nQuestion: {question}\n\nDetailed Answer:"
    )
)

# Rebuild chain with improved prompt -- same retriever and LLM.
qa_chain_v2 = SimpleRetrievalQA(llm=llm, retriever=retriever, prompt=improved_prompt)

# Test on a question that scored low in Cell 15.
test_q = "How do I set task dependencies in Airflow?"
result_v2 = qa_chain_v2.invoke({"query": test_q})

print(f"Improved answer for: '{test_q}'")
print('-' * 60)
print(result_v2["result"])
print()
print("If the answer improved, log this chain as V2:")
print("  1. Set qa_chain = qa_chain_v2")
print("  2. Re-run Cells 5 -> 7 -> 8 to log and register Version 2")

---
## Cell 18 -- Cleanup: Delete the Model Serving Endpoint

The Model Serving endpoint has scale-to-zero and costs nothing when idle.
Delete it after your demo to keep the workspace tidy.

Uncomment and run this cell when fully done with the demo.

Note: Delta Tables and Unity Catalog model registry entries persist after endpoint
deletion. Run the SQL commands shown below to also remove the ingested data.

In [0]:
# UNCOMMENT TO DELETE THE MODEL SERVING ENDPOINT WHEN DONE
# Serving endpoint has scale-to-zero so idle cost is $0.
# Uncomment only when fully done with the demo.

# from databricks.sdk import WorkspaceClient
# w = WorkspaceClient()
# w.serving_endpoints.delete(name=deployment.endpoint_name)
# print(f"Deleted serving endpoint: {deployment.endpoint_name}")

# NOTE: Vector Search endpoint (rag-portfolio-endpoint) charges ~$0.28/hr.
# To delete it, run Cell 19 in databricks_rag_vector_search.ipynb.

# Optional: remove ingested data from Unity Catalog:
#   %sql DROP TABLE IF EXISTS rag_portfolio.doc_search.airflow_docs_chunks;
#   %sql DROP TABLE IF EXISTS rag_portfolio.doc_search.eval_results;

print("Cleanup is commented out for safety.")
print("Uncomment the delete lines above when done with the demo.")

---
## Cell 19 -- Portfolio Summary: What This Project Demonstrates

A reference card of the full V1 + V2 skill set demonstrated by this project.

In [0]:
summary = (
    "=================================================================\n"
    "PORTFOLIO PROJECT: Databricks RAG with Mosaic AI (V1 + V2)\n"
    "=================================================================\n"
    "\n"
    "V1 -- databricks_rag_vector_search.ipynb\n"
    "  - Apache Airflow open-source docs ingestion\n"
    "  - Chunking with RecursiveCharacterTextSplitter\n"
    "  - Delta Table source-of-truth (Unity Catalog, CDF enabled)\n"
    "  - Databricks Vector Search (BGE Large, Delta Sync Index)\n"
    "  - LangChain retrieval chain with custom system prompt\n"
    "  - Batch evaluation output persisted to Delta\n"
    "\n"
    "V2 -- databricks_rag_mosaic_ai_v2.ipynb (this notebook)\n"
    "  - mlflow.pyfunc.ChatModel wrapper (OpenAI-compatible interface)\n"
    "  - MLflow Experiment tracking (versioned runs)\n"
    "  - Unity Catalog model registry (Version 1, Version 2 ...)\n"
    "  - agents.deploy -> serverless endpoint (scale-to-zero, REST API)\n"
    "  - Mosaic AI Review App (shareable no-login chat UI)\n"
    "  - MLflow Tracing (RETRIEVER + LLM spans)\n"
    "  - Agent Evaluation (groundedness, relevance, precision)\n"
    "  - Iterative improvement cycle (V1 -> evaluate -> V2)\n"
    "\n"
    "Skills demonstrated:\n"
    "  - End-to-end RAG: data ingestion to deployed REST API\n"
    "  - MLOps: model versioning, evaluation, iterative improvement\n"
    "  - Cost awareness: scale-to-zero serving, cheapest Foundation Models\n"
    "  - Data engineering + AI engineering (rare combined skill set)\n"
    "================================================================="
)
print(summary)